In [1]:
# Step 1: Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [2]:
# Step 2: Load the California housing dataset
df = pd.read_excel("housing.csv.xlsx")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41,880,129.0,322,126,8.3252,452600,NEAR BAY
1,-122.22,37.86,21,7099,1106.0,2401,1138,8.3014,358500,NEAR BAY
2,-122.24,37.85,52,1467,190.0,496,177,7.2574,352100,NEAR BAY
3,-122.25,37.85,52,1274,235.0,558,219,5.6431,341300,NEAR BAY
4,-122.25,37.85,52,1627,280.0,565,259,3.8462,342200,NEAR BAY


In [3]:
# Step 3: Create income categories for stratified sampling based on median_income
df["income_cat"] = pd.cut(
    df["median_income"],
    bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5]
)

In [4]:
# Step 4: Perform Stratified Shuffle Split (80% train, 20% test)
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_index, test_index in split.split(df, df["income_cat"]):
    strat_train_set = df.loc[train_index].copy()
    strat_test_set = df.loc[test_index].copy()

# Drop temporary 'income_cat' column so the dataset reverts to its original features
for sett in (strat_train_set, strat_test_set):
    sett.drop("income_cat", axis=1, inplace=True)

In [5]:
# Step 5: Separate predictors (housing) and target labels (housing_labels)
# Must be separated before any preprocessing transformations
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
12655,-121.46,38.52,29,3873,797.0,2237,706,2.1736,INLAND
15502,-117.23,33.09,7,5320,855.0,2015,768,6.3373,NEAR OCEAN
2908,-119.04,35.37,44,1618,310.0,667,300,2.8750,INLAND
14053,-117.13,32.75,24,1877,519.0,898,483,2.2264,NEAR OCEAN
20496,-118.70,34.28,27,3536,646.0,1837,580,4.4964,<1H OCEAN


In [6]:
# Step 6: Isolate numerical features
# Drop 'ocean_proximity' because our current pipeline handles numerical transformations
housing_num = housing.drop("ocean_proximity", axis=1)
housing_num.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
12655,-121.46,38.52,29,3873,797.0,2237,706,2.1736
15502,-117.23,33.09,7,5320,855.0,2015,768,6.3373
2908,-119.04,35.37,44,1618,310.0,667,300,2.8750
14053,-117.13,32.75,24,1877,519.0,898,483,2.2264
20496,-118.70,34.28,27,3536,646.0,1837,580,4.4964


In [7]:
# Step 7: Construct the numerical preprocessing Pipeline
# A Pipeline chains multiple sequential transformations:
# 1. 'impute': SimpleImputer fills missing values (using median strategy)
# 2. 'standardize': StandardScaler centers features to mean=0 and std=1
num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("standardize", StandardScaler()),
])

# Rules to remember for Scikit-Learn Pipelines:
# - Takes a list of ('name', transformer) pairs
# - Step names must be unique strings and must NOT contain double underscores '__'
# - All intermediate steps must be transformers (implementing both fit and transform)
# - The final step can be either a transformer or an estimator/predictor

In [8]:
# Step 8: Fit and transform numerical data through the pipeline
# fit_transform() runs the steps sequentially:
#   Output of 'impute' is passed directly into 'standardize'
housing_num_tr = num_pipeline.fit_transform(housing_num)

# Output is a 2D NumPy array with all 8 numerical features imputed and scaled
print("Transformed array shape:", housing_num_tr.shape)
housing_num_tr

Transformed array shape: (16512, 8)


array([[-0.94135046,  1.34743822,  0.02756357, ...,  0.73260236,
         0.55628602, -0.8936472 ],
       [ 1.17178212, -1.19243966, -1.72201763, ...,  0.53361152,
         0.72131799,  1.292168  ],
       [ 0.26758118, -0.1259716 ,  1.22045984, ..., -0.67467519,
        -0.52440722, -0.52543365],
       ...,
       [-1.5707942 ,  1.31001828,  1.53856552, ..., -0.86201341,
        -0.86511838, -0.36547546],
       [-1.56080303,  1.2492109 , -1.1653327 , ..., -0.18974707,
         0.01061579,  0.16826095],
       [-1.28105026,  2.02567448, -0.13148926, ..., -0.71232211,
        -0.79857323, -0.390569  ]], shape=(16512, 8))

In [9]:
# Step 9: Convert the transformed array back into a DataFrame for inspection
# Reuse original column names and row indices from housing_num
housing_tr_df = pd.DataFrame(
    housing_num_tr,
    columns=housing_num.columns,
    index=housing_num.index
)
housing_tr_df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
12655,-0.941350,1.347438,0.027564,0.584777,0.640371,0.732602,0.556286,-0.893647
15502,1.171782,-1.192440,-1.722018,1.261467,0.781561,0.533612,0.721318,1.292168
2908,0.267581,-0.125972,1.220460,-0.469773,-0.545138,-0.674675,-0.524407,-0.525434
14053,1.221738,-1.351474,-0.370069,-0.348652,-0.036367,-0.467617,-0.037297,-0.865929
20496,0.437431,-0.635818,-0.131489,0.427179,0.272790,0.374060,0.220898,0.325752


In [10]:
# Step 10: Verify pipeline output
# 1. Confirm missing values are handled (should be 0):
print("Remaining missing values:", housing_tr_df.isnull().sum().sum())

# 2. Confirm standardization (means ≈ 0.0 and stds ≈ 1.0):
print("\nFeature Means (should be ~0.0):")
print(housing_tr_df.mean().round(4))
print("\nFeature Standard Deviations (should be ~1.0):")
print(housing_tr_df.std().round(4))

Remaining missing values: 0

Feature Means (should be ~0.0):
longitude             0.0
latitude              0.0
housing_median_age    0.0
total_rooms          -0.0
total_bedrooms       -0.0
population           -0.0
households           -0.0
median_income        -0.0
dtype: float64

Feature Standard Deviations (should be ~1.0):
longitude             1.0
latitude              1.0
housing_median_age    1.0
total_rooms           1.0
total_bedrooms        1.0
population            1.0
households            1.0
median_income         1.0
dtype: float64
